In [ ]:
import os
import pandas as pd
import numpy as np
import json

try:
    import xgboost as xgb
except:
    !pip install "xgboost<3"
    import xgboost as xgb

print(f'XGBoost version: {xgb.__version__}')

#### Constants

In [ ]:
str_bucket = 'assessment-alt'
str_task = '05_model'
str_target = 'flt_log_price'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

#### Load Data

In [ ]:
%%time
X_train = pd.read_csv(f's3://{str_bucket}/04_preprocessing/X_train.csv')
X_valid = pd.read_csv(f's3://{str_bucket}/04_preprocessing/X_valid.csv')
X_test = pd.read_csv(f's3://{str_bucket}/04_preprocessing/X_test.csv')
y_train = pd.read_csv(f's3://{str_bucket}/04_preprocessing/y_train.csv')
y_valid = pd.read_csv(f's3://{str_bucket}/04_preprocessing/y_valid.csv')
y_test = pd.read_csv(f's3://{str_bucket}/04_preprocessing/y_test.csv')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print(f'\nFeatures: {X_train.columns.tolist()}')

#### Create DMatrices

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train[str_target])
dvalid = xgb.DMatrix(X_valid, label=y_valid[str_target])
dtest = xgb.DMatrix(X_test, label=y_test[str_target])
print(f'DMatrices created: train={dtrain.num_row()}, valid={dvalid.num_row()}, test={dtest.num_row()}')

#### Define Parameters

In [ ]:
dict_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_lambda': 1.0,
    'reg_alpha': 0.1,
    'seed': 42,
    'verbosity': 0,
}
print('Parameters:')
for str_key, val in dict_params.items():
    print(f'  {str_key}: {val}')

#### Train Model

In [ ]:
%%time
model = xgb.train(
    dict_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=100,
    verbose_eval=100,
)
int_best_iteration = model.best_iteration
print(f'\nBest iteration: {int_best_iteration}')
print(f'Best validation RMSE: {model.best_score:.4f}')

#### Overfitting Check

In [ ]:
arr_pred_train = model.predict(dtrain, iteration_range=(0, int_best_iteration + 1))
arr_pred_valid = model.predict(dvalid, iteration_range=(0, int_best_iteration + 1))
arr_pred_test = model.predict(dtest, iteration_range=(0, int_best_iteration + 1))

flt_rmse_train = np.sqrt(np.mean((y_train[str_target].values - arr_pred_train) ** 2))
flt_rmse_valid = np.sqrt(np.mean((y_valid[str_target].values - arr_pred_valid) ** 2))
flt_rmse_test = np.sqrt(np.mean((y_test[str_target].values - arr_pred_test) ** 2))

df_overfit = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'RMSE (log price)': [round(flt_rmse_train, 4), round(flt_rmse_valid, 4), round(flt_rmse_test, 4)],
})
print(df_overfit.to_string(index=False))

#### Generate Predictions

In [ ]:
# Convert predictions back to actual price scale
arr_price_true = np.expm1(y_test[str_target].values)
arr_price_pred = np.expm1(arr_pred_test)

df_predictions = pd.DataFrame({
    'flt_log_price_true': y_test[str_target].values,
    'flt_log_price_pred': arr_pred_test,
    'flt_price_true': arr_price_true,
    'flt_price_pred': arr_price_pred,
})

print(f'Predictions shape: {df_predictions.shape}')
print(f'\nPredicted price stats:')
print(df_predictions[['flt_price_true', 'flt_price_pred']].describe().round(2))

#### Save Artifacts

In [ ]:
# Save model
model.save_model(f'{str_dirname_output}/xgb_model.json')
print(f'Saved xgb_model.json to {str_dirname_output}')

# Save parameters
with open(f'{str_dirname_output}/params.json', 'w') as f:
    json.dump(dict_params, f, indent=2)
print(f'Saved params.json to {str_dirname_output}')

# Save best iteration
with open(f'{str_dirname_output}/best_iteration.json', 'w') as f:
    json.dump({'best_iteration': int_best_iteration}, f, indent=2)
print(f'Saved best_iteration.json to {str_dirname_output}')

# Save overfitting metrics
dict_overfit = {
    'train_rmse': round(flt_rmse_train, 4),
    'valid_rmse': round(flt_rmse_valid, 4),
    'test_rmse': round(flt_rmse_test, 4),
}
with open(f'{str_dirname_output}/overfitting_metrics.json', 'w') as f:
    json.dump(dict_overfit, f, indent=2)
print(f'Saved overfitting_metrics.json to {str_dirname_output}')

# Save predictions to S3
str_s3_path = f's3://{str_bucket}/{str_task}/test_predictions.csv'
df_predictions.to_csv(str_s3_path, index=False)
print(f'Saved test_predictions.csv to {str_s3_path}')

#### Takeaways

- **XGBoost regression** trained on log-transformed price with early stopping
- **Overfitting check**: compare train/valid/test RMSE — if train << valid/test, the model is overfitting
- **Early stopping** on validation set prevents overfitting by stopping before the model memorizes training data
- **Predictions saved** in both log scale and actual dollar scale for evaluation
- Model artifacts saved locally for reproducibility